# 01b · The n-tables: every site together

Reads the per-site CSVs that `01_fixed_config_n_tables_site.ipynb` writes (`results/fixedcfg_nb_diag/<site>.csv`)
and prints one table per number of images n: the rows are the sites and the Wadden survey-year groups, the
columns the methods. Each method is scored on its own pixels with a value within F: RMSE, bias = median(p − t),
centred RMSE, OLS slope, r, coverage of F, and the seconds of its estimator call; then the centred RMSE on the
common pixels (where every computable method has a value) and their share of F. Then the error against n per
site, one line per method. Last, the diagnostic variants (section 9b of the site notebook), in tables of their
own.

In [ ]:
import glob, os
while not os.path.isdir("pyintertidal") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")                                    # from notebooks/validation/ to the repository root
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = os.environ.get("NB_OUT", "results/fixedcfg_nb_diag")  # where the site notebooks wrote
ORDER = ["MAREA", "NIDEM", "DEA", "E4", "E5a", "E5b", "E5c", "E7", "E12"]
SCORES = {"rmse": "RMSE (m)", "bias": "bias (m)", "crmse": "centred RMSE (m)", "slope": "slope", "r": "r",
          "coverage": "coverage of F", "seconds": "time (s)", "common_crmse": "centred RMSE, common (m)",
          "common_share": "common share of F"}

## 1 · What was run

Every CSV of the folder, and the status of every site row, n and method: *ok*, *not reachable* (fewer than n
screened images on or before the truth date), *not computable* (the method's own rules give no product),
*error*, or *pending truth* (Villaviciosa without its LiDAR).

In [ ]:
files = sorted(glob.glob(f"{OUT}/*.csv"))
d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
methods = [m for m in ORDER if m in set(d.method)]          # the nine methods; the variants are in section 4
print(f"{len(d)} rows from {[os.path.basename(f) for f in files]}")
display(d.pivot_table(index=["site", "n"], columns="method", values="status", aggfunc="first")
          .reindex(columns=methods))

## 2 · One table per n

Rows: site (or Wadden survey-year group) and score; columns: the methods. A row that cannot fill the window is
listed under the table with the number of screened images it has.

In [ ]:
ok = d[d.status == "ok"]
for n in sorted(d.n.unique(), reverse=True):
    print(f"\nn = {n}")
    t = ok[ok.n == n].melt(id_vars=["site", "method"], value_vars=list(SCORES), var_name="score")
    if len(t):
        t["score"] = pd.Categorical(t.score.map(SCORES), categories=list(SCORES.values()), ordered=True)
        t = t.pivot_table(index=["site", "score"], columns="method", values="value", observed=True)
        display(t.reindex(columns=[m for m in methods if m in t.columns]).round(3))
    for _, r in d[(d.n == n) & (d.status != "ok")].drop_duplicates(["site", "status"]).iterrows():
        if r.status in ("not reachable", "pending truth"):
            print(f"  {r.site}: {r.status} ({r.reason})")

## 3 · The error against n

Centred RMSE on each row's pixels F, one panel per site row, one line per method (MAREA in black).

In [ ]:
COLORS = {"MAREA": "#0b0b0b", "NIDEM": "#2a78d6", "DEA": "#eb6834", "E4": "#1baf7a", "E5a": "#eda100",
          "E5b": "#e87ba4", "E5c": "#008300", "E7": "#6250d6", "E12": "#e34948"}
MARKERS = {"MAREA": "o", "NIDEM": "s", "DEA": "D", "E4": "^", "E5a": "v", "E5b": "<", "E5c": ">", "E7": "P",
           "E12": "X"}
rows = sorted(ok.site.unique())
if rows:
    fig, axes = plt.subplots(1, len(rows), figsize=(4.8 * len(rows), 4.4), squeeze=False)
    for ax, s in zip(axes[0], rows):
        for m in methods:
            e = ok[(ok.site == s) & (ok.method == m)].sort_values("n")
            if len(e):
                ax.plot(e.n, e.crmse, marker=MARKERS.get(m, "o"), ms=7, lw=2.6 if m == "MAREA" else 1.6,
                        color=COLORS.get(m), label=e.label.iloc[0])
        ax.set_title(s, loc="left")
        ax.set_xlabel("images in the window, n")
        ax.set_xticks(sorted(ok[ok.site == s].n.unique()))
        ax.grid(alpha=0.3)
    axes[0][0].set_ylabel("centred RMSE (m)")
    axes[0][-1].legend(fontsize=8, loc="best")
    fig.tight_layout()
    fig.savefig(f"{OUT}/curves_crmse.png", dpi=130)

## 4 · The diagnostic variants

Exploratory, post hoc, never replacing a method (section 9b of the site notebook): MAREA against MAREA with no
delay and MAREA with z at NDWI = 0, and LogiTide2DEM (E4), sampled exceedance (E5c) and DEA Intertidal against
themselves with MAREA's per-pixel delay. One table per n: the rows are the site rows and the methods or variants,
the columns RMSE, bias, centred RMSE, slope, coverage of F and seconds, on the row's pixels F. A variant that
failed is listed under its table.

In [ ]:
DIAG = ["MAREA", "MAREA_nodelay", "MAREA_ndwi0", "E4", "E4_delay", "E5c", "E5c_delay", "DEA", "DEA_delay"]
v = ok[ok.method.isin(DIAG)].assign(order=lambda x: x.method.map(DIAG.index))
for n in sorted(v.n.unique(), reverse=True):
    print(f"\nn = {n}")
    display(v[v.n == n].sort_values(["site", "order"]).set_index(["site", "label"])
              [["rmse", "bias", "crmse", "slope", "coverage", "seconds"]].round(3))
    for r in d[(d.n == n) & d.method.isin(DIAG) & d.status.isin(["error", "not computable"])].itertuples():
        print(f"  {r.site} {r.label}: {r.status} ({r.reason})")